# Notebook 02 - Data Quality Assessment

**Project:** Bankruptcy Risk Analytics Platform

## Objective
Assess the quality, integrity, and reliability of the dataset before feature engineering and machine learning.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize']=(10,5)

DATA_PATH='data/raw/american_bankruptcy.csv'
df=pd.read_csv(DATA_PATH)


## 2. Dataset Snapshot

In [ ]:
display(df.head())
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

## 3. Data Types Validation

In [ ]:
dtype_report=pd.DataFrame({
    'Column':df.columns,
    'DataType':df.dtypes.astype(str)
})
display(dtype_report)

## 4. Missing Value Assessment

In [ ]:
missing=df.isna().sum().to_frame('Missing')
missing['Percent']=100*missing['Missing']/len(df)
display(missing)
missing['Percent'].plot(kind='bar',title='Missing Values (%)')
plt.show()

## 5. Duplicate Assessment

In [ ]:
print("Duplicate Rows:",df.duplicated().sum())
print("Duplicate Company-Year Records:",
      df.duplicated(subset=['company_name','year']).sum())

## 6. Target Integrity

In [ ]:
target=df['status_label'].value_counts()
display(target)
target.plot(kind='bar',title='Class Distribution')
plt.show()

print('Class Ratio')
print((target/len(df)*100).round(2))

## 7. Company Integrity

In [ ]:
company=df.groupby('company_name').size()
display(company.describe())

company.hist(bins=30)
plt.title('Observations per Company')
plt.show()

## 8. Year Validation

In [ ]:
print("Year Range:",df['year'].min(),"-",df['year'].max())
df['year'].value_counts().sort_index().plot(marker='o',title='Records by Year')
plt.show()

## 9. Constant & Low Variance Features

In [ ]:
numeric=df.select_dtypes(include='number')
constant=[c for c in numeric.columns if numeric[c].nunique()==1]
low_var=numeric.var().sort_values()

print("Constant Features:",constant)
display(low_var.head(10))

## 10. Invalid Value Checks

In [ ]:
negative_counts=(numeric<0).sum().sort_values(ascending=False)
display(negative_counts.to_frame('Negative Values'))

print("Review whether negative values are valid for each financial variable based on business meaning.")

## 11. Outlier Overview (IQR)

In [ ]:
Q1=numeric.quantile(.25)
Q3=numeric.quantile(.75)
IQR=Q3-Q1

outliers=((numeric<(Q1-1.5*IQR))|(numeric>(Q3+1.5*IQR))).sum()
display(outliers.sort_values(ascending=False).to_frame('Outlier Count'))

outliers.sort_values().plot(kind='barh',title='Outliers by Feature')
plt.show()

## 12. Data Quality Scorecard

In [ ]:
scorecard=pd.DataFrame({
'Check':['Missing Values','Duplicate Rows','Company-Year Duplicates','Target Labels','Year Range'],
'Result':[
'PASS' if df.isna().sum().sum()==0 else 'FAIL',
'PASS' if df.duplicated().sum()==0 else 'REVIEW',
'PASS' if df.duplicated(subset=['company_name','year']).sum()==0 else 'REVIEW',
'PASS' if set(df['status_label'].unique())<=set(['alive','failed','bankrupt']) else 'REVIEW',
'PASS'
]})
display(scorecard)

## 13. Executive Findings

In [ ]:
findings=[
'Dataset completeness is acceptable if no missing values are detected.',
'Class imbalance should be addressed during model development.',
'Outliers are expected in financial datasets and should not be removed blindly.',
'Validate negative values using feature definitions before preprocessing.'
]

for i,f in enumerate(findings,1):
    print(f'{i}. {f}')